# Xunzi Baseline Inference 

Notebook này chạy inference cho **Base Model Xunzi-Qwen3-8B** (không có LoRA Adapter, không có Retrieval) để thiết lập **Baseline Performance**.

**Mục tiêu so sánh**:
- Model: Base (Zero-shot) vs. Fine-tuned (SFT)
- Context: No context vs. Retrieved Context (SPEADO)

**Luồng xử lý**:
1. Load Base Model (Unsloth optimization)
2. Prompt Zero-shot (Chỉ đưa instruction, không đưa ví dụ)
3. Inference Pipeline (Sliding window + Robust alignment như file finetune)
4. Tính Metrics (Seg F1, Punc F1, OSC)

In [1]:
# Cài đặt thư viện (giống file finetune)
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps "xformers" "trl<0.9.0" "accelerate" "bitsandbytes" --quiet
!pip install datasketch jiwer seqeval opencc-python-reimplemented --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 13.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of th

In [2]:
from unsloth import FastLanguageModel
import difflib
import json
import os
import re
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence
import numpy as np
import pandas as pd
import torch
from jiwer import cer
from tqdm.auto import tqdm

@dataclass(frozen=True)
class Config:
    # --- Model paths ---
    # CHỈ DÙNG BASE MODEL, KHÔNG CÓ ADAPTER
    base_model_dir: str = "/kaggle/input/xunzi-qwen3-8b/pytorch/default/1/Xunzi-Qwen3-8B"
    
    # --- Data files ---
    # Test files giống hệt file infer finetune để so sánh công bằng
    test_files: Sequence[str] = (
        "/kaggle/input/evahan-2022/evahan_testa.json",
        "/kaggle/input/evahan-2022/EvaHan_testa_gold.json",
        "/kaggle/input/evahan-2022/evahan_testb.json",
        "/kaggle/input/evahan-2022/EvaHan_testb_gold.json",
    )

    # --- Inference params ---
    max_seq_length: int = 1024
    chunk_size: int = 512
    overlap_size: int = 128
    
    # Decoding strategy
    num_beams: int = 1

    # Whitelist punctuation
    PUNC_SET = set([
        '"', '(', ')', ',', '-', ':', '、', '。', '〈', '〉', '《', '》',
        '「', '」', '『', '』', '〔', '〕', '！', '（', '）', '，', '．', '：', '；', '？'
    ])

    puncs: frozenset = frozenset(PUNC_SET)
    seed: int = 42

CFG = Config()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-01-04 16:51:29.382899: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767545489.564236      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767545489.614610      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767545490.018823      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767545490.018865      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767545490.018868      24 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not import trl.trainer.ddpo_trainer: Failed to import trl.trainer.ddpo_trainer because of the following error (look up to see its traceback):
Failed to import trl.models.modeling_sd_base because of the following error (look up to see its traceback):
Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.ip_adapter because of the following error (look up to see its traceback):
/usr/local/lib/python3.12/dist-packages/xformers/flash_attn_3/_C.so: undefined symbol: _ZNK3c106SymInt22maybe_as_int_slow_pathEv


## 2) Utilities & Metrics (Giữ nguyên)
Phần này phải giữ nguyên 100% so với `xunzi-finetune-infer.ipynb` để đảm bảo logic tính điểm là nhất quán.

In [3]:
# --- Text Cleaning Utils ---
def strip_pos_tags(text: str) -> str:
    no_tags = re.sub(r"/[a-z]+", "", text)
    no_tags = re.sub(r"\s+", " ", no_tags).strip()
    return no_tags

def strip_punc_and_spaces(text: str) -> str:
    return "".join(ch for ch in text if ch not in CFG.puncs and ch != " ")

def seg_only(text: str) -> str:
    return "".join(ch for ch in text if ch not in CFG.puncs)

def punc_only(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text.replace(" ", "")

def clean_model_output(text: str) -> str:
    if "### Response:" in text:
        text = text.split("### Response:")[-1]
    elif "<|im_start|>assistant" in text:
        text = text.split("<|im_start|>assistant")[-1]
    pattern = r"<think>.*?</think>"
    cleaned_text = re.sub(pattern, "", text, flags=re.DOTALL)
    cleaned_text = cleaned_text.replace("<|im_end|>", "").replace("<|endoftext|>", "")
    return cleaned_text.strip()

# --- Metrics Class (Imported from finetune-infer to ensure consistency) ---
from seqeval.metrics import f1_score

class Metrics:
    # ... (Copy logic Metrics từ file cũ để đảm bảo tính chính xác)
    PUNC_MAP = {
        "，": "COMMA", ",": "COMMA", "、": "PAUSE", "。": "PERIOD", "．": "DOT",
        "！": "EXCLAM", "？": "QUEST", "；": "SEMICOLON", "：": "COLON", ":": "COLON", "-": "HYPHEN",
        "“": "QUOTE_L", "”": "QUOTE_R", "\"": "QUOTE", "（": "PAREN_L", "(": "PAREN_L",
        "）": "PAREN_R", ")": "PAREN_R", "《": "BOOK_L", "》": "BOOK_R", "〈": "ANGLE_L",
        "〉": "ANGLE_R", "「": "CORNER_L", "」": "CORNER_R", "『": "CORNER_W_L",
        "』": "CORNER_W_R", "〔": "TORTOISE_L", "〕": "TORTOISE_R",
    }
    PUNC_PRIORITY = {"PERIOD": 3, "QUEST": 3, "EXCLAM": 3, "COMMA": 2, "SEMICOLON": 2, "PAUSE": 2}

    @staticmethod
    def _seg_tags_from_seg_only(seg_text: str) -> List[str]:
        tags = []
        start_of_word = True
        for ch in seg_text:
            if ch == " ":
                start_of_word = True
                continue
            tags.append("B-W" if start_of_word else "I-W")
            start_of_word = False
        return tags

    @staticmethod
    def _punc_tags_from_punc_only(punc_text: str, puncs: set) -> List[str]:
        tags = []
        last_raw_index = -1
        for ch in punc_text:
            if ch in puncs:
                if last_raw_index >= 0:
                    new_name = Metrics.PUNC_MAP.get(ch, "PUNC")
                    new_tag = f"B-{new_name}"
                    current_tag = tags[last_raw_index]
                    if current_tag == "O":
                        tags[last_raw_index] = new_tag
                    else:
                        current_name = current_tag[2:] if current_tag.startswith("B-") else current_tag
                        if int(Metrics.PUNC_PRIORITY.get(new_name, 1)) > int(Metrics.PUNC_PRIORITY.get(current_name, 1)):
                            tags[last_raw_index] = new_tag
                continue
            tags.append("O")
            last_raw_index += 1
        return tags

    @staticmethod
    def _joint_tags_from_full_text(full_text: str, puncs: set) -> List[str]:
        s = re.sub(r"\s+", " ", (full_text or "").strip())
        tags: List[str] = []
        word_start = None
        word_len = 0
        best_punc_name = "NOPUNC"
        best_punc_pri = 0

        def flush_word():
            nonlocal word_start, word_len, best_punc_name, best_punc_pri
            if word_start is None: return
            ent_type = f"W__{best_punc_name}"
            for i in range(word_len):
                prefix = "B" if i == 0 else "I"
                tags[word_start + i] = f"{prefix}-{ent_type}"
            word_start, word_len, best_punc_name, best_punc_pri = None, 0, "NOPUNC", 0

        for ch in s:
            if ch == " ":
                flush_word(); continue
            if ch in puncs:
                if word_start is not None:
                    name = Metrics.PUNC_MAP.get(ch, "PUNC")
                    pri = int(Metrics.PUNC_PRIORITY.get(name, 1))
                    if pri > best_punc_pri:
                        best_punc_pri = pri; best_punc_name = name
                continue
            if word_start is None:
                word_start = len(tags); word_len = 0; best_punc_name = "NOPUNC"; best_punc_pri = 0
            tags.append("O"); word_len += 1
        flush_word()
        return tags

    @staticmethod
    def compute(pred_full: str, gt_full: str, puncs: set) -> Dict[str, float]:
        pred_raw, gt_raw = strip_punc_and_spaces(pred_full), strip_punc_and_spaces(gt_full)
        try: fidelity = max(0.0, 1.0 - float(cer(gt_raw, pred_raw)))
        except: fidelity = 0.0

        p_seg, t_seg = Metrics._seg_tags_from_seg_only(seg_only(pred_full)), Metrics._seg_tags_from_seg_only(seg_only(gt_full))
        p_punc, t_punc = Metrics._punc_tags_from_punc_only(punc_only(pred_full), puncs), Metrics._punc_tags_from_punc_only(punc_only(gt_full), puncs)
        p_joint, t_joint = Metrics._joint_tags_from_full_text(pred_full, puncs), Metrics._joint_tags_from_full_text(gt_full, puncs)

        min_len = min(len(p_seg), len(t_seg), len(p_punc), len(t_punc), len(p_joint), len(t_joint))
        if min_len != len(p_seg) or min_len != len(t_seg): fidelity = 0.0 # Hard penalty length mismatch
        
        seg_f1 = float(f1_score([t_seg[:min_len]], [p_seg[:min_len]])) if t_seg else 0.0
        punc_f1 = float(f1_score([t_punc[:min_len]], [p_punc[:min_len]])) if t_punc else 0.0
        punc_and_seg_f1 = float(f1_score([t_joint[:min_len]], [p_joint[:min_len]])) if t_joint else 0.0
        
        return {
            "seg_f1": seg_f1,
            "punc_f1": punc_f1, 
            "punc_and_seg_f1": punc_and_seg_f1, 
            "fidelity": fidelity, 
            "osc": fidelity * seg_f1 * punc_f1
        }

## 3) Baseline Inference Pipeline (Zero-shot, No Retrieval)

Các thay đổi chính so với file finetune:
1. **Prompt Template**: Đơn giản hóa, loại bỏ các tham chiếu đến `example_section` (ví dụ) và `sys_ref_instruction`.
2. **InferencePipeline**: Loại bỏ hoàn toàn logic `retriever`. Mọi input đều được xử lý ở chế độ Zero-shot.
3. **Model Loading**: Chỉ load Base Model qua Unsloth, không load adapter.

In [4]:
# --- ZERO-SHOT PROMPT TEMPLATE ---
filtered_punct = [c for c in CFG.puncs if len(c.encode("utf-8")) < 4 and c.isprintable()]
valid_puncts_str = " ".join(sorted(filtered_punct))

# Prompt Baseline: Chỉ chứa instruction, không có chỗ cho reference example
PROMPT_TEMPLATE = """<|im_start|>system
你是一个专业的古代汉语智能助手，专注于古文的“分词”与“标点”任务。

任务要求：
1. **分词规范**：依照EvaHan学术标准，词与词之间使用半角空格 " " 分隔。
2. **标点规范**：仅使用指定标点集合：{valid_puncts}，标点附着在词后，并在标点后添加空格。
3. **原文忠实**：严禁修改、增加或删除原文中的任何汉字，必须保持字符级别的绝对一致。
<|im_end|>
<|im_start|>user
### 待处理文本：
{input}
<|im_end|>
<|im_start|>assistant
"""

def build_baseline_prompt(input_text: str) -> str:
    return PROMPT_TEMPLATE.format(
        valid_puncts=valid_puncts_str,
        input=input_text
    ) + "\n"


def _load_base_model_only():
    """Chỉ load Base Model, không load Adapter."""
    print("[Model] Loading Base Model (Unsloth)... ")
    model, tokenizer = FastLanguageModel.from_pretrained(
        CFG.base_model_dir,
        max_seq_length=CFG.max_seq_length,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer


def _load_json_list_files(paths: Sequence[str]) -> List[dict]:
    data = []
    for p in list(paths):
        if not os.path.exists(p): continue
        with open(p, "r", encoding="utf-8") as f:
            if p.lower().endswith(".jsonl"):
                for line in f: data.append(json.loads(line))
            else:
                data.extend(json.load(f))
    return data

class BaselinePipeline:
    """
    Simplified pipeline for Baseline:
    - No retriever.
    - Zero-shot prompting.
    - Same sliding window & robust alignment logic.
    """
    ALLOWED_PUNCS = set(CFG.puncs) | {" "}

    def __init__(self, model, tokenizer, chunk_size=1024, overlap=256):
        self.model = model
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.max_seq_length = CFG.max_seq_length
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self._eos_ids = [self.tokenizer.eos_token_id]

    def robust_align(self, raw_chunk: str, model_output: str) -> str:
        # Hàm này cực kỳ quan trọng để đảm bảo fidelity = 1.0 cho phần character
        model_output = clean_model_output(model_output)
        matcher = difflib.SequenceMatcher(None, raw_chunk, model_output)
        out = []
        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            if tag == "equal":
                out.append(raw_chunk[i1:i2])
            elif tag == "insert":
                # Chỉ chấp nhận chèn dấu câu/space, không chấp nhận chèn chữ lạ
                out.append("".join(c for c in model_output[j1:j2] if c in self.ALLOWED_PUNCS))
            elif tag == "replace":
                # Giữ nguyên chữ gốc, chỉ lấy thêm dấu câu từ phần replace
                out.append(raw_chunk[i1:i2] + "".join(c for c in model_output[j1:j2] if c in self.ALLOWED_PUNCS))
            elif tag == "delete":
                # Không cho phép model xóa chữ gốc
                out.append(raw_chunk[i1:i2])
        return "".join(out)

    def _predict_chunk(self, text: str) -> str:
        # Zero-shot prompt
        prompt = build_baseline_prompt(text)
        
        inputs = self.tokenizer([prompt], return_tensors="pt", add_special_tokens=False).to(self.device)
        prompt_len = inputs["input_ids"].shape[1]

        # Nếu prompt dài quá context window (hiếm khi xảy ra vì đã sliding window)
        if prompt_len > self.max_seq_length:
            # Fallback thô: cắt bớt input (nhưng sliding window bên dưới đã handle việc này)
            pass 

        max_new_tokens = min(2048, int(len(text) * 1.5) + 64)
        
        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                do_sample=False,
                num_beams=CFG.num_beams,
                max_new_tokens=max_new_tokens,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self._eos_ids,
            )
        
        gen_ids = out_ids[0][prompt_len:]
        decoded = self.tokenizer.decode(gen_ids, skip_special_tokens=False)
        return self.robust_align(text, decoded)

    def _slice_by_raw_count(self, text_with_punc: str, skip_raw: int, keep_raw: int) -> str:
        if keep_raw <= 0: return ""
        raw_counter, kept_raw, started = 0, 0, False
        out = []
        for ch in text_with_punc:
            if ch in self.ALLOWED_PUNCS:
                if started: out.append(ch)
                continue
            if raw_counter < skip_raw:
                raw_counter += 1; continue
            if kept_raw < keep_raw:
                started = True; out.append(ch); kept_raw += 1; raw_counter += 1
                continue
            break
        return "".join(out)

    def predict_long_text(self, long_raw_text: str) -> str:
        # Logic Sliding Window giữ nguyên từ file gốc
        n = len(long_raw_text)
        if n == 0: return ""
        
        chunk_size = self.chunk_size
        overlap = self.overlap
        stride = chunk_size - 2 * overlap

        if n <= chunk_size:
            return self._predict_chunk(long_raw_text)

        final_parts = []
        
        # First chunk
        first_raw = long_raw_text[0:chunk_size]
        first_pred = self._predict_chunk(first_raw)
        final_parts.append(self._slice_by_raw_count(first_pred, 0, chunk_size - overlap))

        current_raw_idx = chunk_size - overlap
        while current_raw_idx < n:
            start = current_raw_idx - overlap
            end = min(n, start + chunk_size)
            chunk_raw = long_raw_text[start:end]
            pred = self._predict_chunk(chunk_raw)
            
            is_last = (end == n)
            keep_r = len(chunk_raw) - overlap if is_last else stride
            
            core_part = self._slice_by_raw_count(pred, overlap, keep_r)
            final_parts.append(core_part)
            current_raw_idx += keep_r
            if is_last: break
            
        return "".join(final_parts)

## 4) Chạy thực nghiệm Baseline
Chạy trên tập test (giống tập test của file finetune) và lưu kết quả vào `baseline_results.csv`.

In [5]:
def run_baseline(
    output_csv: str = "baseline_results.csv",
    num_samples: Optional[int] = None
) -> pd.DataFrame:
    
    # 1. Load Model
    model, tokenizer = _load_base_model_only()
    
    # 2. Init Pipeline (No Retrieval)
    pipeline = BaselinePipeline(
        model=model, 
        tokenizer=tokenizer, 
        chunk_size=CFG.chunk_size, 
        overlap=CFG.overlap_size
    )

    # 3. Load Data
    data = _load_json_list_files(CFG.test_files)
    if num_samples:
        data = data[:num_samples]
    
    print(f"[Infer] Running BASELINE on {len(data)} samples...")
    
    rows = []
    for sample in tqdm(data, desc="Baseline"):
        gt_full = strip_pos_tags(sample["output"])
        original = strip_punc_and_spaces(gt_full)
        
        # Predict
        pred_full = pipeline.predict_long_text(original)
        
        # Metrics
        m = Metrics.compute(pred_full, gt_full, CFG.puncs)
        
        rows.append({
            "original": original,
            "gt_full": gt_full,
            "pred_full": pred_full,
            "seg_f1": m["seg_f1"],
            "punc_f1": m["punc_f1"],
            "punc_and_seg_f1": m["punc_and_seg_f1"],
            "fidelity": m["fidelity"],
            "osc": m["osc"]
        })
    
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Saved to {output_csv}")
    
    # Print Summary
    print("\n=== BASELINE METRICS ===")
    print(f"Seg F1: {df['seg_f1'].mean():.4f}")
    print(f"Punc F1: {df['punc_f1'].mean():.4f}")
    print(f"Joint F1: {df['punc_and_seg_f1'].mean():.4f}")
    print(f"OSC: {df['osc'].mean():.4f}")
    
    return df

# Chạy thực nghiệm
df_baseline = run_baseline(num_samples=None) 

[Model] Loading Base Model (Unsloth)... 
==((====))==  Unsloth 2026.1.1: Fast Qwen3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[Infer] Running BASELINE on 4117 samples...


Baseline:   0%|          | 0/4117 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:159: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(


Saved to baseline_results.csv

=== BASELINE METRICS ===
Seg F1: 0.3415
Punc F1: 0.2655
Joint F1: 0.3333
OSC: 0.0342
